# Statistics Analysis for InstanceNorm1d

Investigating why streaming audio is quieter than reference - collecting mean/variance statistics from all normalization layers.

In [32]:
# Cell 1: Imports
import os
os.chdir('/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro')

In [33]:
# Cell 2: PyTorch and core imports
import math
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from IPython.display import Audio, display
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [31]:
# Cell 3: Load Kokoro model and config
from kokoro import KModel, KPipeline
from kokoro.model import KModelForONNX

# Paths
config_file = 'checkpoints/config.json'
checkpoint_path = 'checkpoints/kokoro-v1_0.pth'

# Load model (KModel handles loading from path)
kmodel = KModel(config=config_file, model=checkpoint_path, disable_complex=True).to(device)
model = KModelForONNX(kmodel).eval()

# Load voice
voice_name = 'af_heart'
style = torch.load(f'checkpoints/voices/{voice_name}.pt', weights_only=True).to(device)

# Load config for later use
with open(config_file) as f:
    config = json.load(f)

# Create pipeline for text processing
pipeline = KPipeline(lang_code='a', model=kmodel, device=device)

print(f"Model loaded on {device}. Voice: {voice_name}")

/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model loaded on cuda. Voice: af_heart


In [5]:
# Cell 4: Streaming Generator Classes (Clean Implementation)

def get_padding(kernel_size, dilation=1):
    return int((kernel_size * dilation - dilation) / 2)

def init_weights(m, mean=0.0, std=0.01):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        m.weight.data.normal_(mean, std)


class AdaIN1d_Streaming(nn.Module):
    """AdaIN with first-chunk statistics for streaming."""
    
    def __init__(self, style_dim, num_features):
        super().__init__()
        self.num_features = num_features
        self.fc = nn.Linear(style_dim, num_features * 2)
        self.eps = 1e-5
        self.register_buffer('cached_mean', torch.zeros(1, num_features, 1))
        self.register_buffer('cached_var', torch.ones(1, num_features, 1))
        self.register_buffer('stats_initialized', torch.tensor(False))
        
    def reset_stats(self):
        """Reset cached statistics for new utterance."""
        self.cached_mean.zero_()
        self.cached_var.fill_(1)
        self.stats_initialized.fill_(False)
        
    def forward(self, x, s):
        h = self.fc(s)
        h = h.view(h.size(0), h.size(1), 1)
        gamma, beta = torch.chunk(h, chunks=2, dim=1)
        
        # Compute and cache statistics on first call
        if not self.stats_initialized:
            with torch.no_grad():
                self.cached_mean.copy_(x.mean(dim=2, keepdim=True))
                self.cached_var.copy_(x.var(dim=2, keepdim=True, unbiased=False))
                self.stats_initialized.fill_(True)
        
        normalized = (x - self.cached_mean) / torch.sqrt(self.cached_var + self.eps)
        return (1 + gamma) * normalized + beta


class AdaINResBlock1_Streaming(nn.Module):
    """AdaINResBlock1 for streaming with cached statistics."""
    
    def __init__(self, channels, kernel_size=3, dilation=(1, 3, 5), style_dim=64):
        super().__init__()
        self.convs1 = nn.ModuleList([
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=dilation[0],
                                  padding=get_padding(kernel_size, dilation[0]))),
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=dilation[1],
                                  padding=get_padding(kernel_size, dilation[1]))),
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=dilation[2],
                                  padding=get_padding(kernel_size, dilation[2])))
        ])
        self.convs1.apply(init_weights)
        self.convs2 = nn.ModuleList([
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=1,
                                  padding=get_padding(kernel_size, 1))),
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=1,
                                  padding=get_padding(kernel_size, 1))),
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=1,
                                  padding=get_padding(kernel_size, 1)))
        ])
        self.convs2.apply(init_weights)
        
        self.adain1 = nn.ModuleList([
            AdaIN1d_Streaming(style_dim, channels),
            AdaIN1d_Streaming(style_dim, channels),
            AdaIN1d_Streaming(style_dim, channels),
        ])
        self.adain2 = nn.ModuleList([
            AdaIN1d_Streaming(style_dim, channels),
            AdaIN1d_Streaming(style_dim, channels),
            AdaIN1d_Streaming(style_dim, channels),
        ])
        self.alpha1 = nn.ParameterList([nn.Parameter(torch.ones(1, channels, 1)) for _ in range(3)])
        self.alpha2 = nn.ParameterList([nn.Parameter(torch.ones(1, channels, 1)) for _ in range(3)])

    def reset_stats(self):
        for adain in self.adain1:
            adain.reset_stats()
        for adain in self.adain2:
            adain.reset_stats()

    def forward(self, x, s):
        for c1, c2, n1, n2, a1, a2 in zip(self.convs1, self.convs2, self.adain1, self.adain2, self.alpha1, self.alpha2):
            xt = n1(x, s)
            xt = xt + (1 / a1) * (torch.sin(a1 * xt) ** 2)  # Snake1D
            xt = c1(xt)
            xt = n2(xt, s)
            xt = xt + (1 / a2) * (torch.sin(a2 * xt) ** 2)  # Snake1D
            xt = c2(xt)
            x = xt + x
        return x


class StreamingGenerator(nn.Module):
    """
    Streaming vocoder generator for chunk-based audio synthesis.
    
    Usage:
        1. Call upsample_f0() once with full F0 curve
        2. Call precompute_x_source() once with full har
        3. Call generate_chunk() for each chunk of x
    """
    
    def __init__(self, original_generator, config):
        super().__init__()
        self.num_kernels = original_generator.num_kernels
        self.num_upsamples = original_generator.num_upsamples
        self.upsample_rates = config['istftnet']['upsample_rates']
        self.total_upsample = math.prod(self.upsample_rates)
        self.post_n_fft = original_generator.post_n_fft
        self.gen_istft_hop_size = config['istftnet']['gen_istft_hop_size']
        
        # Copy shared modules (no modification needed)
        self.m_source = original_generator.m_source
        self.f0_upsamp = original_generator.f0_upsamp
        self.noise_convs = original_generator.noise_convs
        self.ups = original_generator.ups
        self.conv_post = original_generator.conv_post
        self.reflection_pad = original_generator.reflection_pad
        self.stft = original_generator.stft
        
        # Create streaming versions of AdaIN blocks
        style_dim = config['style_dim']
        resblock_kernel_sizes = config['istftnet']['resblock_kernel_sizes']
        resblock_dilation_sizes = config['istftnet']['resblock_dilation_sizes']
        upsample_initial_channel = config['istftnet']['upsample_initial_channel']
        
        # noise_res blocks
        self.noise_res = nn.ModuleList()
        for i in range(len(self.ups)):
            c_cur = upsample_initial_channel // (2 ** (i + 1))
            kernel = 7 if i + 1 < len(self.upsample_rates) else 11
            self.noise_res.append(AdaINResBlock1_Streaming(c_cur, kernel, [1, 3, 5], style_dim))
        
        # Main resblocks
        self.resblocks = nn.ModuleList()
        for i in range(len(self.ups)):
            ch = upsample_initial_channel // (2 ** (i + 1))
            for k, d in zip(resblock_kernel_sizes, resblock_dilation_sizes):
                self.resblocks.append(AdaINResBlock1_Streaming(ch, k, d, style_dim))
        
        # Copy weights from original
        self._copy_weights(original_generator)
        
    def _copy_weights(self, original):
        """Copy weights from original generator."""
        for orig, stream in zip(original.noise_res, self.noise_res):
            for i in range(len(orig.convs1)):
                stream.convs1[i].load_state_dict(orig.convs1[i].state_dict())
                stream.convs2[i].load_state_dict(orig.convs2[i].state_dict())
            for i in range(len(orig.alpha1)):
                stream.alpha1[i].data.copy_(orig.alpha1[i].data)
                stream.alpha2[i].data.copy_(orig.alpha2[i].data)
            for i in range(len(orig.adain1)):
                stream.adain1[i].fc.load_state_dict(orig.adain1[i].fc.state_dict())
                stream.adain2[i].fc.load_state_dict(orig.adain2[i].fc.state_dict())
        
        for orig, stream in zip(original.resblocks, self.resblocks):
            for i in range(len(orig.convs1)):
                stream.convs1[i].load_state_dict(orig.convs1[i].state_dict())
                stream.convs2[i].load_state_dict(orig.convs2[i].state_dict())
            for i in range(len(orig.alpha1)):
                stream.alpha1[i].data.copy_(orig.alpha1[i].data)
                stream.alpha2[i].data.copy_(orig.alpha2[i].data)
            for i in range(len(orig.adain1)):
                stream.adain1[i].fc.load_state_dict(orig.adain1[i].fc.state_dict())
                stream.adain2[i].fc.load_state_dict(orig.adain2[i].fc.state_dict())
    
    def reset_stats(self):
        """Reset all cached statistics for new utterance."""
        for block in self.noise_res:
            block.reset_stats()
        for block in self.resblocks:
            block.reset_stats()
            
    def upsample_f0(self, f0):
        """Upsample F0 and compute harmonic source."""
        with torch.no_grad():
            f0_up = self.f0_upsamp(f0[:, None]).transpose(1, 2)
            har_source, _, _ = self.m_source(f0_up)
            har_source = har_source.transpose(1, 2).squeeze(1)
            har_spec, har_phase = self.stft.transform(har_source)
            har = torch.cat([har_spec, har_phase], dim=1)
        return har
    
    def precompute_x_source(self, har_full, s):
        """Precompute x_source for all stages using full harmonic signal."""
        x_source_stages = []
        with torch.no_grad():
            for i in range(self.num_upsamples):
                x_source = self.noise_convs[i](har_full)
                x_source = self.noise_res[i](x_source, s)
                x_source_stages.append(x_source)
        return x_source_stages
    
    def generate_chunk(self, x_chunk, s, x_source_stages, x_start_idx):
        """Generate audio for a single chunk."""
        x = x_chunk
        
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, negative_slope=0.1)
            x = self.ups[i](x)
            if i == self.num_upsamples - 1:
                x = self.reflection_pad(x)
            
            # Slice x_source for this chunk
            src_start = x_start_idx * self.upsample_rates[0] if i == 0 else x_start_idx * self.total_upsample
            src_end = src_start + x.shape[-1]
            x_source = x_source_stages[i][:, :, src_start:src_end]
            
            min_len = min(x.shape[-1], x_source.shape[-1])
            x = x[:, :, :min_len] + x_source[:, :, :min_len]
            
            xs = None
            for j in range(self.num_kernels):
                idx = i * self.num_kernels + j
                if xs is None:
                    xs = self.resblocks[idx](x, s)
                else:
                    xs = xs + self.resblocks[idx](x, s)
            x = xs / self.num_kernels
        
        x = F.leaky_relu(x)
        x = self.conv_post(x)
        
        spec = torch.exp(x[:, :self.post_n_fft // 2 + 1, :])
        phase = torch.sin(x[:, self.post_n_fft // 2 + 1:, :])
        return self.stft.inverse(spec, phase)


print("Streaming generator classes defined.")

Streaming generator classes defined.


In [21]:
# Cell 5: Metrics computation functions

def compute_audio_metrics(ref_audio, test_audio):
    """Compute quality metrics between reference and test audio."""
    ref = ref_audio.float()
    test = test_audio.float()
    
    min_len = min(len(ref), len(test))
    ref = ref[:min_len]
    test = test[:min_len]
    
    # MSE and MAE
    mse = torch.mean((ref - test) ** 2).item()
    mae = torch.mean(torch.abs(ref - test)).item()
    
    # SNR
    signal_power = torch.mean(ref ** 2)
    noise_power = torch.mean((ref - test) ** 2)
    snr_db = 10 * torch.log10(signal_power / (noise_power + 1e-10)).item()
    
    # Correlation
    ref_centered = ref - ref.mean()
    test_centered = test - test.mean()
    correlation = (torch.sum(ref_centered * test_centered) / 
                  (torch.sqrt(torch.sum(ref_centered**2)) * torch.sqrt(torch.sum(test_centered**2)) + 1e-10)).item()
    
    # Max absolute error
    max_abs_error = torch.max(torch.abs(ref - test)).item()
    
    return {
        'mse': mse,
        'mae': mae,
        'snr_db': snr_db,
        'correlation': correlation,
        'max_abs_error': max_abs_error
    }


def compute_spike_metrics(ref_audio, test_audio, threshold_factor=5.0):
    """Compute spike/outlier metrics."""
    ref = ref_audio.numpy() if hasattr(ref_audio, 'numpy') else ref_audio
    test = test_audio.numpy() if hasattr(test_audio, 'numpy') else test_audio
    
    min_len = min(len(ref), len(test))
    diff = np.abs(ref[:min_len] - test[:min_len])
    
    # Spike detection: points where error > threshold_factor * median error
    median_diff = np.median(diff)
    threshold = threshold_factor * median_diff
    spikes = diff > threshold
    
    num_spikes = np.sum(spikes)
    spike_ratio = num_spikes / len(diff)
    
    # Get spike locations and magnitudes
    spike_indices = np.where(spikes)[0]
    spike_magnitudes = diff[spikes] if num_spikes > 0 else np.array([])
    
    # Percentile metrics
    p95_error = np.percentile(diff, 95)
    p99_error = np.percentile(diff, 99)
    
    return {
        'num_spikes': int(num_spikes),
        'spike_ratio': spike_ratio,
        'max_spike_magnitude': float(np.max(spike_magnitudes)) if num_spikes > 0 else 0.0,
        'mean_spike_magnitude': float(np.mean(spike_magnitudes)) if num_spikes > 0 else 0.0,
        'p95_error': float(p95_error),
        'p99_error': float(p99_error),
        'threshold': float(threshold)
    }


print("Metrics functions defined.")

Metrics functions defined.


In [19]:
# Cell 6: Simplified end-to-end TTS function (using model internals directly)

CHUNK_SIZE = 64  # x-frames per chunk
SAMPLE_RATE = 24000


def text_to_phonemes(pipeline_obj, text):
    """Convert text to phonemes using g2p."""
    if pipeline_obj.lang_code in 'ab':
        _, tokens = pipeline_obj.g2p(text)
        for gs, ps, tks in pipeline_obj.en_tokenize(tokens):
            if ps:
                return ps
    else:
        ps, _ = pipeline_obj.g2p(text)
        return ps
    return []


def text_to_audio_streaming(text, pipeline_obj, kmodel_obj, streaming_gen_obj, voice_style, voice_name, chunk_size=CHUNK_SIZE):
    """
    Simplified end-to-end text to audio using streaming generator.
    
    Args:
        text: Input text
        pipeline_obj: KPipeline for text processing
        kmodel_obj: KModel for TTS
        streaming_gen_obj: StreamingGenerator instance
        voice_style: Pre-loaded voice tensor [510, 1, 256]
        voice_name: Voice name (for logging)
    
    Returns:
        streaming_audio: Audio generated via streaming
        reference_audio: Audio generated via standard forward pass
        timing: Dict with timing information
    """
    timing = {}
    device = next(kmodel_obj.parameters()).device
    
    # Step 1: Text to phonemes
    t0 = time.perf_counter()
    phonemes = text_to_phonemes(pipeline_obj, text)
    if len(phonemes) > 510:
        phonemes = phonemes[:510]
    
    # Convert to input_ids
    input_ids = list(filter(lambda i: i is not None, map(lambda p: kmodel_obj.vocab.get(p), phonemes)))
    input_ids = torch.LongTensor([[0, *input_ids, 0]]).to(device)
    
    # Get voice style tensor (select based on phoneme length)
    ref_s = voice_style[len(phonemes) - 1]  # [1, 256]
    speed = torch.IntTensor([1]).to(device)
    
    timing['preprocessing'] = time.perf_counter() - t0
    
    # Step 2: Run model forward to get intermediate tensors
    t0 = time.perf_counter()
    with torch.no_grad():
        # Call model internals step by step (from forward_with_tokens)
        text_mask = torch.ones(input_ids.shape, dtype=torch.float32, device=device)
        
        # BERT encoding
        bert_dur = kmodel_obj.bert(input_ids, attention_mask=text_mask)
        d_en = kmodel_obj.bert_encoder(bert_dur).transpose(-1, -2)
        
        # Predictor
        s_pred = ref_s[:, 128:]  # [1, 128] for predictor
        d = kmodel_obj.predictor.text_encoder(d_en, s_pred, text_mask)
        x_lstm, _ = kmodel_obj.predictor.lstm(d)
        
        # Duration prediction
        duration = kmodel_obj.predictor.duration_proj(x_lstm)
        duration = text_mask * torch.sigmoid(duration).sum(axis=-1) / speed
        pred_dur = torch.round(duration).clamp(min=1).squeeze()
        
        # Expand phoneme embeddings
        boundaries = torch.cumsum(pred_dur, dim=0)
        values = torch.arange(boundaries[-1], device=device)
        expanded_indices = torch.sum(boundaries.unsqueeze(1) <= values.unsqueeze(0), dim=0)
        
        en = torch.index_select(d.transpose(-1, -2), 2, expanded_indices)
        en_t, _ = kmodel_obj.predictor.shared(en.transpose(-1, -2))
        
        # F0 prediction
        F0_pred, N_pred = kmodel_obj.predictor.F0Ntrain(en_t, s_pred)
        
        # Text encoder
        t_en = kmodel_obj.text_encoder(input_ids, text_mask)
        asr = torch.index_select(t_en, 2, expanded_indices)
        
        # Decoder style
        s_decoder = ref_s[:, :128]  # [1, 128] for decoder
        
        # Now replicate decoder processing up to generator call
        decoder = kmodel_obj.decoder
        F0 = decoder.F0_conv(F0_pred.unsqueeze(1))
        N = decoder.N_conv(N_pred.unsqueeze(1))
        
        x = torch.cat([asr, F0, N], axis=1)
        x = decoder.encode(x, s_decoder)
        
        asr_res = decoder.asr_res(asr)
        res = True
        for i, block in enumerate(decoder.decode):
            if res:
                x = torch.cat([x, asr_res, F0, N], axis=1)
            x = block(x, s_decoder)
            if block.upsample_type != "none":
                res = False
        
        # Now x is ready for the generator!
        # x shape should be [1, 512, T] where T is audio frames / total_upsample
        
    timing['model_forward'] = time.perf_counter() - t0
    
    # Step 3: Generate reference audio using full generator
    t0 = time.perf_counter()
    with torch.no_grad():
        ref_audio_tensor = decoder.generator(x, s_decoder, F0_pred)
        ref_audio = ref_audio_tensor.squeeze().cpu()
    timing['reference_gen'] = time.perf_counter() - t0
    
    # Step 4: Streaming generation
    streaming_gen_obj.reset_stats()
    
    t0 = time.perf_counter()
    with torch.no_grad():
        # Precompute
        har_full = streaming_gen_obj.upsample_f0(F0_pred)
        x_source_stages = streaming_gen_obj.precompute_x_source(har_full, s_decoder)
    timing['precompute'] = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    audio_chunks = []
    with torch.no_grad():
        for x_start in range(0, x.shape[2], chunk_size):
            x_end = min(x_start + chunk_size, x.shape[2])
            x_chunk = x[:, :, x_start:x_end]
            audio_chunk = streaming_gen_obj.generate_chunk(x_chunk, s_decoder, x_source_stages, x_start)
            audio_chunks.append(audio_chunk.squeeze().cpu())
    timing['streaming_gen'] = time.perf_counter() - t0
    
    streaming_audio = torch.cat(audio_chunks)
    
    return streaming_audio, ref_audio, timing


print(f"Simplified TTS function defined. Chunk size: {CHUNK_SIZE}")


Simplified TTS function defined. Chunk size: 64


In [7]:
# Cell 7: Create streaming generator

# Access the Generator from the decoder
original_generator = kmodel.decoder.generator

# Create streaming version
streaming_gen = StreamingGenerator(original_generator, config).to(device).eval()

print("Streaming generator created.")

Streaming generator created.


In [8]:
# Cell 8: Test sentences (50 diverse sentences)

TEST_SENTENCES = [
    # Short sentences (5-10 words)
    "Hello, how are you today?",
    "The quick brown fox jumps.",
    "Please pass the salt.",
    "I love sunny days.",
    "Where is the nearest station?",
    
    # Medium sentences (10-20 words)
    "The weather forecast predicts rain for the entire weekend ahead.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The committee will meet next Tuesday to discuss the proposed budget changes.",
    "She practiced piano for three hours every day to prepare for the concert.",
    "The ancient castle stood majestically on the hilltop overlooking the peaceful valley.",
    
    # Long sentences (20+ words)
    "After years of dedicated research and countless experiments, the team finally achieved a breakthrough in renewable energy technology.",
    "The old bookshop on the corner of Main Street has been serving the community for over fifty years with rare and antique books.",
    "When the sun sets behind the mountains, the sky transforms into a beautiful canvas of orange, pink, and purple hues.",
    "The international conference brought together experts from various fields to discuss climate change solutions and sustainable development strategies.",
    "Despite facing numerous challenges and setbacks throughout the project, the team remained motivated and successfully delivered the results on time.",
    
    # Questions
    "Have you ever wondered what lies beyond the visible universe?",
    "Could you please explain how the new system works in detail?",
    "What would you do if you won a million dollars tomorrow?",
    "Why do birds fly south for the winter season every year?",
    "How long have you been studying artificial intelligence and machine learning?",
    
    # Exclamations and emphasis
    "What an absolutely amazing performance that was tonight!",
    "I can't believe we finally made it to the top of the mountain!",
    "This is the most delicious chocolate cake I have ever tasted!",
    "Congratulations on your incredible achievement and hard work!",
    "The fireworks display was spectacular and truly breathtaking!",
    
    # Technical/formal language
    "The algorithm processes data in parallel using distributed computing resources.",
    "Pursuant to the agreement, all parties shall maintain strict confidentiality.",
    "The pharmaceutical compound demonstrated significant efficacy in clinical trials.",
    "Neural networks require substantial computational resources for training deep models.",
    "The quantum computer achieved unprecedented processing speeds in benchmark tests.",
    
    # Conversational/informal
    "Hey, did you catch the game last night? It was incredible!",
    "So basically, we just need to finish this by Friday, no big deal.",
    "I'm totally going to that new restaurant everyone's been talking about.",
    "You know what, I think we should just take a break and relax.",
    "Man, traffic was absolutely insane this morning on the highway.",
    
    # Numbers and lists
    "The recipe calls for two cups of flour, three eggs, and one teaspoon of vanilla.",
    "Our flight departs at seven forty-five in the morning from gate B twelve.",
    "The company reported revenues of three point five billion dollars last quarter.",
    "Please call me at five five five, one two three four for more information.",
    "The temperature will range between sixty-five and seventy-two degrees today.",
    
    # Emotional content
    "I'm so incredibly proud of everything you've accomplished this year.",
    "The news brought tears of joy to everyone in the room.",
    "Sometimes life throws unexpected challenges, but we must stay strong.",
    "Her gentle words brought comfort to those who were struggling.",
    "The heartwarming reunion touched the hearts of millions around the world.",
    
    # Descriptive/narrative
    "The moonlight cast silver shadows across the tranquil garden path.",
    "Autumn leaves danced gracefully in the cool evening breeze.",
    "The aroma of freshly baked bread filled the cozy kitchen.",
    "Crystal clear waters reflected the towering snow-capped mountains above.",
    "The bustling marketplace overflowed with colorful fabrics and exotic spices.",
]

print(f"Defined {len(TEST_SENTENCES)} test sentences.")

Defined 50 test sentences.


In [23]:
# Cell 9: Run evaluation on all test sentences

results = []

print("Running evaluation on 50 test sentences...")
print("=" * 80)

for idx, text in enumerate(TEST_SENTENCES):
    try:
        # Generate audio
        streaming_audio, ref_audio, timing = text_to_audio_streaming(
            text, pipeline, kmodel, streaming_gen, style, voice_name
        )
        
        # Compute metrics
        audio_metrics = compute_audio_metrics(ref_audio, streaming_audio)
        spike_metrics = compute_spike_metrics(ref_audio, streaming_audio)
        
        # Store results
        result = {
            'idx': idx,
            'text': text[:50] + '...' if len(text) > 50 else text,
            'text_full': text,
            'num_tokens': len(text.split()),  # Approximate token count
            'audio_length_ms': len(ref_audio) / SAMPLE_RATE * 1000,
            **audio_metrics,
            **spike_metrics,
            **timing
        }
        results.append(result)
        
        # Print progress
        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1}/{len(TEST_SENTENCES)} sentences...")
            
    except Exception as e:
        print(f"Error on sentence {idx}: {e}")
        results.append({
            'idx': idx,
            'text': text[:50] + '...' if len(text) > 50 else text,
            'error': str(e)
        })

print(f"\nCompleted evaluation of {len(results)} sentences.")


Running evaluation on 50 test sentences...
Processed 10/50 sentences...
Processed 20/50 sentences...
Processed 30/50 sentences...
Processed 40/50 sentences...
Processed 50/50 sentences...

Completed evaluation of 50 sentences.


In [24]:
# Cell 10: Create results DataFrame and display summary

df = pd.DataFrame(results)

# Filter out errors
df_valid = df[~df['snr_db'].isna()].copy()

print("=" * 80)
print("OVERALL METRICS SUMMARY")
print("=" * 80)

metrics_cols = ['snr_db', 'correlation', 'max_abs_error', 'num_spikes', 'spike_ratio', 'p95_error', 'p99_error']

print(f"\n{'Metric':<20} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
print("-" * 68)

for col in metrics_cols:
    if col in df_valid.columns:
        mean_val = df_valid[col].mean()
        std_val = df_valid[col].std()
        min_val = df_valid[col].min()
        max_val = df_valid[col].max()
        print(f"{col:<20} {mean_val:>12.4f} {std_val:>12.4f} {min_val:>12.4f} {max_val:>12.4f}")

print(f"\nTotal sentences processed: {len(df_valid)}")
print(f"Total audio duration: {df_valid['audio_length_ms'].sum() / 1000:.1f} seconds")

OVERALL METRICS SUMMARY

Metric                       Mean          Std          Min          Max
--------------------------------------------------------------------
snr_db                    -2.6274       6.7460     -18.9518       6.4585
correlation                0.5566       0.2118       0.1321       0.8848
max_abs_error              2.2214       3.1663       0.3161      15.4324
num_spikes             20038.1400    5061.5843   10786.0000   33647.0000
spike_ratio                0.1928       0.0564       0.1076       0.3980
p95_error                  0.1362       0.1108       0.0480       0.6302
p99_error                  0.4216       0.4854       0.0882       2.0888

Total sentences processed: 50
Total audio duration: 231.8 seconds


In [25]:
# Cell 11: Identify and display worst 5 sentences

print("=" * 80)
print("TOP 5 WORST PERFORMING SENTENCES (by SNR)")
print("=" * 80)

worst_5 = df_valid.nsmallest(5, 'snr_db')

for i, (_, row) in enumerate(worst_5.iterrows()):
    print(f"\n{i+1}. Sentence #{row['idx']}")
    print(f"   Text: {row['text_full']}")
    print(f"   SNR: {row['snr_db']:.2f} dB | Correlation: {row['correlation']:.4f}")
    print(f"   Spikes: {row['num_spikes']} ({row['spike_ratio']*100:.2f}%)")
    print(f"   Max Error: {row['max_abs_error']:.4f} | P99 Error: {row['p99_error']:.4f}")
    print(f"   Audio Length: {row['audio_length_ms']:.0f}ms")

TOP 5 WORST PERFORMING SENTENCES (by SNR)

1. Sentence #28
   Text: Neural networks require substantial computational resources for training deep models.
   SNR: -18.95 dB | Correlation: 0.1362
   Spikes: 24554 (17.41%)
   Max Error: 15.4324 | P99 Error: 2.0888
   Audio Length: 5875ms

2. Sentence #20
   Text: What an absolutely amazing performance that was tonight!
   SNR: -18.37 dB | Correlation: 0.1911
   Spikes: 14157 (16.73%)
   Max Error: 11.0168 | P99 Error: 1.7251
   Audio Length: 3525ms

3. Sentence #45
   Text: The moonlight cast silver shadows across the tranquil garden path.
   SNR: -17.48 dB | Correlation: 0.1321
   Spikes: 22005 (21.08%)
   Max Error: 8.9718 | P99 Error: 1.8913
   Audio Length: 4350ms

4. Sentence #46
   Text: Autumn leaves danced gracefully in the cool evening breeze.
   SNR: -14.72 dB | Correlation: 0.1847
   Spikes: 18228 (20.81%)
   Max Error: 8.1329 | P99 Error: 1.1850
   Audio Length: 3650ms

5. Sentence #19
   Text: How long have you been studying 

In [27]:
# Cell 12: Audio comparison for worst sentences

print("=" * 80)
print("AUDIO COMPARISON FOR WORST 5 SENTENCES")
print("=" * 80)

for i, (_, row) in enumerate(worst_5.iterrows()):
    idx = row['idx']
    text = TEST_SENTENCES[idx]
    
    # Regenerate audio for playback
    streaming_audio, ref_audio, _ = text_to_audio_streaming(
        text, pipeline, kmodel, streaming_gen, style, voice_name
    )
    
    print(f"\n--- Sentence #{idx} (SNR: {row['snr_db']:.2f} dB) ---")
    print(f"Text: {text[:80]}..." if len(text) > 80 else f"Text: {text}")
    
    print("\nReference:")
    display(Audio(ref_audio.numpy(), rate=SAMPLE_RATE))
    
    print("Streaming:")
    display(Audio(streaming_audio.numpy(), rate=SAMPLE_RATE))


AUDIO COMPARISON FOR WORST 5 SENTENCES

--- Sentence #28 (SNR: -18.95 dB) ---
Text: Neural networks require substantial computational resources for training deep mo...

Reference:


Streaming:



--- Sentence #20 (SNR: -18.37 dB) ---
Text: What an absolutely amazing performance that was tonight!

Reference:


Streaming:



--- Sentence #45 (SNR: -17.48 dB) ---
Text: The moonlight cast silver shadows across the tranquil garden path.

Reference:


Streaming:



--- Sentence #46 (SNR: -14.72 dB) ---
Text: Autumn leaves danced gracefully in the cool evening breeze.

Reference:


Streaming:



--- Sentence #19 (SNR: -14.20 dB) ---
Text: How long have you been studying artificial intelligence and machine learning?

Reference:


Streaming:


In [28]:
# Cell 13: Full results table

print("=" * 80)
print("FULL RESULTS TABLE")
print("=" * 80)

# Display key columns
display_cols = ['idx', 'text', 'audio_length_ms', 'snr_db', 'correlation', 'num_spikes', 'spike_ratio', 'max_abs_error']
df_display = df_valid[display_cols].copy()
df_display = df_display.sort_values('snr_db')

# Format for display
df_display['snr_db'] = df_display['snr_db'].round(2)
df_display['correlation'] = df_display['correlation'].round(4)
df_display['spike_ratio'] = (df_display['spike_ratio'] * 100).round(2)
df_display['max_abs_error'] = df_display['max_abs_error'].round(4)
df_display['audio_length_ms'] = df_display['audio_length_ms'].round(0).astype(int)

df_display.columns = ['#', 'Text', 'Length(ms)', 'SNR(dB)', 'Corr', 'Spikes', 'Spike%', 'MaxErr']

print(df_display.to_string(index=False))

FULL RESULTS TABLE
 #                                                  Text  Length(ms)  SNR(dB)   Corr  Spikes  Spike%  MaxErr
28 Neural networks require substantial computational ...        5875   -18.95 0.1362   24554   17.41 15.4324
20 What an absolutely amazing performance that was to...        3525   -18.37 0.1911   14157   16.73 11.0168
45 The moonlight cast silver shadows across the tranq...        4350   -17.48 0.1321   22005   21.08  8.9718
46 Autumn leaves danced gracefully in the cool evenin...        3650   -14.72 0.1847   18228   20.81  8.1329
19 How long have you been studying artificial intelli...        4600   -14.20 0.2392   17044   15.44  7.8190
34 Man, traffic was absolutely insane this morning on...        4125   -13.48 0.2704   17883   18.06  4.2579
33 You know what, I think we should just take a break...        3550   -13.47 0.2067   20374   23.91  5.5782
 1                            The quick brown fox jumps.        2150   -11.03 0.2853   16363   31.71  3.9988
